## Silver Layer: Clean & Join Sales + Store Data

Reads raw Bronze tables and applies cleaning, type casting, and null handling
before joining sales transactions with store metadata.

In [0]:
df_sales = spark.table("retail_project.bronze.sales_raw")
df_store = spark.table("retail_project.bronze.store_raw")

print("Sales:", df_sales.count(), "rows")
print("Store:", df_store.count(), "rows")

### Fix data types
Cast `Date` from string to a proper date type for downstream time-series operations.

In [0]:
from pyspark.sql.functions import col, to_date

df_sales_clean = df_sales.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

df_sales_clean.printSchema()

### Standardize StateHoliday
Real values are `0` (no holiday), `a` (public holiday), `b` (Easter), `c` (Christmas).
Add a binary `IsStateHoliday` flag for simpler use as a model feature,
while keeping the original category for more granular analysis.

In [0]:
from pyspark.sql.functions import when, col

df_sales_clean = df_sales_clean.withColumn(
    "IsStateHoliday",
    when(col("StateHoliday") != "0", 1).otherwise(0)
)

df_sales_clean.groupBy("StateHoliday", "IsStateHoliday").count().show()

### Handle nulls in store metadata
- `CompetitionDistance`: 3 nulls → filled with a large placeholder (999999),
  implying no meaningful nearby competition.
- `CompetitionOpenSinceMonth/Year`: 354 nulls → filled with 0, with a separate
  `has_competition_open_date` flag to preserve the "unknown" signal.
- `Promo2SinceWeek/Year`, `PromoInterval`: null whenever `Promo2 = 0`
  (store never opted into the program) — this is expected, not missing data,
  so filled with 0 / "None" as a not-applicable placeholder.

In [0]:
from pyspark.sql.functions import when, col, coalesce, lit

df_store_clean = (
    df_store
    .withColumn(
        "has_competition_open_date",
        when(col("CompetitionOpenSinceYear").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "CompetitionDistance",
        when(col("CompetitionDistance").isNull(), 999999).otherwise(col("CompetitionDistance"))
    )
    .fillna({"CompetitionOpenSinceMonth": 0, "CompetitionOpenSinceYear": 0})
    .withColumn("Promo2SinceWeek", coalesce(col("Promo2SinceWeek"), lit(0)))
    .withColumn("Promo2SinceYear", coalesce(col("Promo2SinceYear"), lit(0)))
    .withColumn("PromoInterval", coalesce(col("PromoInterval"), lit("None")))
)

display(df_store_clean)

### Join sales with store metadata
Left join on `Store` — expect row count to match `sales_clean` exactly
(1,017,209 rows), since each sale should map to exactly one store record.

In [0]:
df_joined = df_sales_clean.join(df_store_clean, on="Store", how="left")

print("Joined rows:", df_joined.count())
display(df_joined)